<a href="https://colab.research.google.com/github/jrpallapati/qdrant-hybrid-search/blob/main/Hybrid_Search_%26_Retrieval_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hybrid Search & Retrieval**

## **Problem Statement**

Build a **hybrid search & retrieval system** over biomedical research abstracts using the [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) dataset. Your system must combine **two complementary retrieval strategies** — dense (semantic) and sparse (lexical) — and fuse their results using **Reciprocal Rank Fusion (RRF)** so that the final ranking outperforms either method alone.

You will:

1. Load & clean a subset of PubMedQA  
2. Construct rich, searchable document text from structured fields  
3. Generate **dense** embeddings and **sparse** embeddings
4. Index documents in **Qdrant** with both vector types  
5. Implement a **hybrid search** function that queries both indexes in parallel  
6. Implement **RRF** to merge and re-rank the two result lists  
7. Test and compare all three retrieval modes (dense-only, sparse-only, hybrid)

### Architecture

```
                        ┌─── Dense (BGE-Large) ──────┐
Query ──► Embed Query ──┤                            ├──► RRF Fusion ──► Final Ranked Results         
                        └─── Sparse (SPLADE) ────────┘
```

### Key Concepts

- **Dense retrieval** excels at *semantic* / conceptual matching (e.g., "physical activity" ↔ "exercise")  
- **Sparse retrieval** excels at *exact* / lexical matching (e.g., drug names like "metformin")  
- **Hybrid search** combines both to cover each method's blind spots  
- **RRF** merges ranked lists without needing score normalization — a document found by *both* methods is promoted



## **💡 Tips**




**Document construction matters.** Think carefully about *which* fields to include in the document text and *which* to exclude. The `question` field is what users will search with — including it in the document would allow trivial self-matching and inflate your results.



## **Steps**

### Step 1: Setup & Installation
Install the required packages: `qdrant-client`, `fastembed`, `datasets`, `transformers`. Import all necessary modules.

You will need:
- `QdrantClient` and models from `qdrant_client.models` (`Distance`, `SparseVector`, `PointStruct`, `QueryRequest`, `SparseIndexParams`, `SparseVectorParams`, `VectorParams`, `ScoredPoint`)
- `SparseTextEmbedding` and `TextEmbedding` from `fastembed`
- `load_dataset` from `datasets`
- `pandas`, `numpy`, `json`


In [1]:
# Write your code here
!pip install -qU qdrant-client fastembed datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.1 MB/s eta 0:00:00


In [2]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

FastEmbed version: 0.8.0


### Step 2: Load & Explore the PubMedQA Dataset
Load the `pqa_artificial` split of `qiaojin/PubMedQA`. Print total instances and feature names. Then select a **subset of 100** instances to work with and convert to a pandas DataFrame.


In [13]:
dataset = load_dataset("qiaojin/PubMedQA", name="pqa_artificial", split="train")

In [16]:
dataset = dataset.select(range(120))

In [17]:
print(f"Total query-product pairs: {len(dataset)}")
print(f"\nFeatures: {dataset.column_names}")

Total query-product pairs: 120

Features: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


### Step 3: Data Cleaning & Exploration
Before building documents, inspect the data:
- Check for duplicate `pubid` values  
- Check for missing values and empty strings in `question` and `long_answer`  
- Print the distribution of `final_decision` (yes / no / maybe)

This step ensures you understand what you're working with.


In [22]:
# Write your code here
source_df = dataset.to_pandas()

In [26]:
source_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   pubid           120 non-null    int32 
 1   question        120 non-null    object
 2   context         120 non-null    object
 3   long_answer     120 non-null    object
 4   final_decision  120 non-null    object
dtypes: int32(1), object(4)
memory usage: 4.3+ KB


In [27]:
df = source_df.drop_duplicates(subset="pubid")

In [28]:
df.head()

,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) in...,{'contexts': ['Chronic rhinosinusitis (CRS) is...,"As ILC2s are elevated in patients with CRSwNP,...",yes
1,25433161,Does vagus nerve contribute to the development...,{'contexts': ['Phosphatidylethanolamine N-meth...,Neuronal signals via the hepatic vagus nerve c...,yes
2,25445714,Does psammaplin A induce Sirtuin 1-dependent a...,{'contexts': ['Psammaplin A (PsA) is a natural...,PsA significantly inhibited MCF-7/adr cells pr...,yes
3,25431941,Is methylation of the FGFR2 gene associated wi...,{'contexts': ['This study examined links betwe...,We identified a novel biologically plausible c...,yes
4,25432519,Do tumor-infiltrating immune cell profiles and...,{'contexts': ['Tumor microenvironment immunity...,Breast cancer immune cell subpopulation profil...,yes


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   pubid           120 non-null    int32 
 1   question        120 non-null    object
 2   context         120 non-null    object
 3   long_answer     120 non-null    object
 4   final_decision  120 non-null    object
dtypes: int32(1), object(4)
memory usage: 4.3+ KB


In [30]:
df.iloc[0]

,0
pubid,25429730
question,Are group 2 innate lymphoid cells ( ILC2s ) in...
context,{'contexts': ['Chronic rhinosinusitis (CRS) is...
long_answer,"As ILC2s are elevated in patients with CRSwNP,..."
final_decision,yes


### Step 4: Construct the Document Text
Write a function that combines:
1. **Context paragraphs** — prefixed with their section label, e.g. `[BACKGROUND] ...`  
2. **Long answer** (conclusion) — prefixed with `[CONCLUSION]`  

**Important:** Do **NOT** include the `question` field in the document text — that's what users will query with.

Apply this function to create a new `combined_text` column in your DataFrame.


In [35]:
df["combined_text"] = (
    "[BACKGROUND]" + df["context"].apply(lambda x: " ".join(x["contexts"])) +
    "\n[CONCLUSION]" + df["long_answer"]
)

In [37]:
df["combined_text"]

,combined_text
0,[BACKGROUND]Chronic rhinosinusitis (CRS) is a ...
1,[BACKGROUND]Phosphatidylethanolamine N-methylt...
2,[BACKGROUND]Psammaplin A (PsA) is a natural pr...
3,[BACKGROUND]This study examined links between ...
4,[BACKGROUND]Tumor microenvironment immunity is...
...,...
115,[BACKGROUND]Transposable elements form a signi...
116,[BACKGROUND]Restenosis after vascular interven...
117,[BACKGROUND]The development of human leukocyte...
118,[BACKGROUND]To conduct a systematic review of ...


In [38]:
pd.set_option('display.max_colwidth', None)
print(df['combined_text'].iloc[0])
pd.reset_option('display.max_colwidth')

[BACKGROUND]Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated. The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease. A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotype, endotype, inflamm

### Step 5: Load Embedding Models
Initialize two models from FastEmbed:
- **Sparse model:** `prithvida/Splade_PP_en_v1` — using `SparseTextEmbedding`
- **Dense model:** `BAAI/bge-large-en-v1.5` — using `TextEmbedding`

Write helper functions `make_sparse_embedding(texts)` and `make_dense_embedding(texts)` that wrap each model's `.embed()` method.


In [39]:
# Write your code here
sparse_model_name = "prithvida/Splade_PP_en_v1"
dense_model_name = "BAAI/bge-large-en-v1.5"

In [40]:
# This triggers model download on first run (~500MB sparse + ~1.2GB dense)
sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=32)

print("✅ Both models loaded!")

/tmp/ipykernel_5676/927011117.py:2: DeprecationWarning: The right spelling is prithivida/Splade_PP_en_v1. Support of this name will be removed soon, please fix the model_name
  sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Both models loaded!


In [41]:
# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=32))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

### Step 6: Generate Embeddings
Embed all  documents using both models. Store results as new columns `sparse_embedding` and `dense_embedding` in the DataFrame.



In [43]:
# Write your code here
combined_text = df["combined_text"].tolist()
print(f"Embedding {len(combined_text)} combined text...")

Embedding 120 combined text...


In [44]:
%%time
print("⏳ Generating sparse embeddings (SPLADE)...")
df["sparse_embedding"] = make_sparse_embedding(combined_text)
print("✅ Done!")

⏳ Generating sparse embeddings (SPLADE)...
✅ Done!
CPU times: user 54.7 s, sys: 5.78 s, total: 1min
Wall time: 1min 1s


In [46]:
%%time
print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(combined_text)
print("✅ Done!")

⏳ Generating dense embeddings (BGE-Large)...
✅ Done!
CPU times: user 11min 8s, sys: 5.21 s, total: 11min 13s
Wall time: 11min 15s


### Step 7: Create the Qdrant Collection
Initialize an in-memory `QdrantClient`. Create a collection named `"pubmedqa"` configured with:
- A **dense vector** config named `"text-dense"` — size 1024, cosine distance  
- A **sparse vector** config named `"text-sparse"`


In [47]:
# Write your code here
client = QdrantClient(":memory:")
collection_name = "pubmedqa"

In [48]:
client.create_collection(
    collection_name,
    # Dense vector config: 1024-dim with cosine similarity
    vectors_config={
        "text-dense": VectorParams(
            size=1024,             # Must match model output dimension
            distance=Distance.COSINE,  # Cosine similarity for dense vectors
        )
    },
    # Sparse vector config: no fixed size needed (it's dynamic)
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,    # Keep in RAM for speed
            )
        )
    },
)

print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")

✅ Collection 'pubmedqa' created with dense + sparse vector support


### Step 8: Prepare & Upload Points
Write a function `make_points(df)` that converts each DataFrame row into a `PointStruct` containing:
- **id:** row index  
- **payload:** `text`, `pubid`, `question`, `final_decision`  
- **vectors:** both `text-dense` (list) and `text-sparse` (`SparseVector` with indices and values)

Then upsert all points into the collection.


In [50]:
# Write your code here
def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    combined_text = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(combined_text, sparse_vectors, dense_vectors)
    ):
        # Convert sparse embedding to Qdrant's SparseVector format
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "pubid": rows[idx]["pubid"],
                "question": rows[idx]["question"],
                "final_decision": rows[idx]["final_decision"],
            },
            vector={
                "text-sparse": sparse_vector,   # Sparse vector under its named key
                "text-dense": dense_vector.tolist(),  # Dense vector under its named key
            },
        )
        points.append(point)
    return points


points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

Prepared 120 points for indexing


In [51]:
# Upload to Qdrant
client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")

✅ 120 points indexed in Qdrant!


### Step 9: Implement Hybrid Search
Write a `search(query_text, top_k=10)` function that:
1. Embeds the query with **both** models  
2. Runs dense and sparse queries in **parallel** using `client.query_batch_points()`  
3. Returns both result lists: `[dense_results, sparse_results]`


In [52]:
# Write your code here
def search(query_text: str, top_k: int = 10):
    """
    Perform hybrid search: run dense AND sparse search in parallel,
    return both result lists.
    """
    # Step 1: Embed the query with BOTH models
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    # Step 2: Fire two searches in a single batch request
    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            # Search 1: Dense (semantic similarity)
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            # Search 2: Sparse (lexical matching)
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )

    # Extract the point lists from QueryResponse objects
    return [search_results[0].points, search_results[1].points]

### Step 10: Test Search Queries
Test your search with **different query types** and compare dense vs. sparse results:

For each query, print the top 5 results from both dense and sparse, showing scores and text previews.


In [59]:
# Write your code here
query_text = "statins reduce atrial fibrillation"

search_results = search(query_text)

dense_results, sparse_results = search_results[0], search_results[1]

print(f"Query: '{query_text}'")
print(f"\n{'='*60}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(dense_results[:10]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")

print(f"\n{'='*60}")
print(f"SPARSE (Lexical/SPLADE) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(sparse_results[:10]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}")


Query: 'statins reduce atrial fibrillation'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.6471] [BACKGROUND]The goal of this study was to identify genetic determinants of plasm
  2. [Score: 0.6311] [BACKGROUND]Limited data exist on the risk of developing cardiac sarcoidosis (CS
  3. [Score: 0.6092] [BACKGROUND]Obese patients with idiopathic pulmonary fibrosis (IPF) have higher 
  4. [Score: 0.6078] [BACKGROUND]In recent years, the relationship between physical activity (PA) and
  5. [Score: 0.6064] [BACKGROUND]The etiology of reduced left ventricular (LV) ejection fraction afte
  6. [Score: 0.6038] [BACKGROUND]The aim of this study was to determine the association between 25-hy
  7. [Score: 0.6031] [BACKGROUND]Low serum levels of vitamin D have been associated with depression i
  8. [Score: 0.6006] [BACKGROUND]To evaluate pazopanib eye drops in subjects with active subfoveal ch
  9. [Score: 0.6006] [BACKGROUND]Elevated lipoprotein-associated phospholipase A2 (Lp-PLA2) levels ar
  1

### Step 11: Implement Reciprocal Rank Fusion (RRF)
Implement two functions:

1. `rank_list(search_result)` — converts Qdrant `ScoredPoint` results into `(id, rank)` pairs (1-indexed)  
2. `rrf(rank_lists, k=60, default_rank=1000)` — takes multiple rank lists and returns a single fused list sorted by RRF score

**RRF formula:** For each item, sum `1 / (k + rank)` across all lists. Items not present in a list get `default_rank`.


In [55]:
# Write your code here
def rrf(rank_lists, alpha=60, default_rank=1000):
    """
    Reciprocal Rank Fusion (RRF).

    Takes multiple rank lists and produces a single fused ranking.
    Only considers RANK POSITION, not raw scores — making it safe to
    combine results from different scoring systems.

    Args:
        rank_lists: List of [(item_id, rank), ...] lists
        alpha: Smoothing constant (default=60, from Cormack et al. 2009)
        default_rank: Rank assigned to items not in a list (high = penalty)

    Returns:
        Sorted list of (item_id, rrf_score) tuples
    """
    all_items = set(item for rank_list in rank_lists for item, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    # Matrix: rows = items, cols = rank lists, filled with default_rank
    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    # RRF formula: sum of 1/(alpha + rank) across all lists
    rrf_scores = np.sum(1.0 / (alpha + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)  # Descending
    sorted_items = [(list(item_to_index.keys())[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

### Step 12: Apply RRF & Analyze Results
Apply RRF to fuse dense and sparse results from one of your test queries. Print the final ranked list, showing for each result:
- Its rank  
- Whether it came from **BOTH**, **Dense only**, or **Sparse only**  
- A text preview

**Analyze:** Do you see documents promoted that appeared in both lists? Are there any results that only one method found?


In [56]:
def rank_list(search_result: list[ScoredPoint]):
    """Convert Qdrant search results to (id, rank) pairs."""
    return [(point.id, rank + 1) for rank, point in enumerate(search_result)]


# Convert both result lists to rank lists
dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

print("Dense ranks:", dense_rank_list[:5], "...")
print("Sparse ranks:", sparse_rank_list[:5], "...")

Dense ranks: [(49, 1), (114, 2), (10, 3), (43, 4), (101, 5)] ...
Sparse ranks: [(112, 1), (110, 2), (77, 3), (8, 4), (14, 5)] ...


In [60]:
# Fuse the rankings!
rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 HYBRID SEARCH RESULTS for: '{query_text}'")
print(f"{'='*70}")

# Retrieve full records for the fused results
fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

# Create a lookup by ID
record_by_id = {r.id: r for r in fused_records}

for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = record_by_id[item_id]
    title = record.payload['text'].split('\n')[0]

    # Check if this item was in dense, sparse, or both
    in_dense = any(id == item_id for id, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] {title}")


🏆 HYBRID SEARCH RESULTS for: 'statins reduce atrial fibrillation'
   1. [BOTH        ] [BACKGROUND]Recent epidemiological evidence suggests that modifying lifestyle by increasing physical activity could be a non-pharmacological approach to improving symptoms and slowing disease progression in Alzheimer's disease and other tauopathies. Previous studies have shown that exercise reduces tau hyperphosphorylation, however, it is not known whether exercise reduces the accumulation of soluble or insoluble tau aggregates and neurofibrillary tangles, which are both neuropathological hallmarks of neurodegenerative tauopathy. In this study, 7-month old P301S tau transgenic mice were subjected to 12-weeks of forced treadmill exercise and evaluated for effects on motor function and tau pathology at 10 months of age. Exercise improved general locomotor and exploratory activity and resulted in significant reductions in full-length and hyperphosphorylated tau in the spinal cord and hippocampus as wel